# 2. Train and compare models

This notebook trains regression models (Ridge, Lasso, KNN and a neural network) to predict the monthly rent in Kenya shillings and classification models (Naive Bayes, KNN and a neural network) to label it as affordable, average or expensive. The best model of each type is saved for the Flask backend.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
MODEL_DIR = ROOT / "backend" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "clean_rent.csv")
FEATURES = ["bedrooms", "size", "city", "furnishing"]
X = df[FEATURES]
y_reg = df["rent"].to_numpy()
q33, q66 = np.quantile(y_reg, [1 / 3, 2 / 3])
y_clf = np.select([y_reg <= q33, y_reg <= q66], ["affordable", "average"], default="expensive")

print(f"Rows: {len(df)}")
print(f"Affordable <= {q33:,.0f}, Average <= {q66:,.0f}, Expensive > {q66:,.0f}")
print(pd.Series(y_clf).value_counts())

Rows: 4963
Affordable <= 15,500, Average <= 87,500, Expensive > 87,500
affordable    1714
expensive     1654
average       1595
Name: count, dtype: int64


In [2]:
X_train, X_test, y_train, y_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), ["bedrooms", "size"]),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["city", "furnishing"]),
        ]
    )

def make_pipeline(estimator):
    return Pipeline([("preprocess", make_preprocessor()), ("model", estimator)])

regressors = {
    "Ridge": TransformedTargetRegressor(
        regressor=make_pipeline(Ridge(alpha=10)), func=np.log1p, inverse_func=np.expm1
    ),
    "Lasso": TransformedTargetRegressor(
        regressor=make_pipeline(Lasso(alpha=0.001, max_iter=20000)), func=np.log1p, inverse_func=np.expm1
    ),
    "KNN": make_pipeline(KNeighborsRegressor(n_neighbors=15, weights="distance")),
    "Neural Network": make_pipeline(
        MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1500, random_state=42, early_stopping=True)
    ),
}

classifiers = {
    "Naive Bayes": make_pipeline(GaussianNB()),
    "KNN": make_pipeline(KNeighborsClassifier(n_neighbors=15, weights="distance")),
    "Neural Network": make_pipeline(
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1500, random_state=42, early_stopping=True)
    ),
}

print("Models ready:", len(regressors), "regressors and", len(classifiers), "classifiers")

Models ready: 4 regressors and 3 classifiers


In [3]:
regression_results = []
for name, model in regressors.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    regression_results.append(
        {
            "model": name,
            "r2": round(float(r2_score(y_test, prediction)), 4),
            "mae": round(float(mean_absolute_error(y_test, prediction)), 2),
            "rmse": round(float(np.sqrt(mean_squared_error(y_test, prediction))), 2),
        }
    )

regression_table = pd.DataFrame(regression_results).sort_values("r2", ascending=False)
regression_table

,model,r2,mae,rmse
3,Neural Network,0.9495,13446.33,23788.90
1,Lasso,0.9312,13848.99,27777.64
2,KNN,0.9287,15154.67,28288.80
0,Ridge,0.9234,14315.05,29315.62


In [4]:
classification_results = []
for name, model in classifiers.items():
    model.fit(X_train, yc_train)
    prediction = model.predict(X_test)
    classification_results.append(
        {
            "model": name,
            "accuracy": round(float(accuracy_score(yc_test, prediction)), 4),
            "f1_weighted": round(float(f1_score(yc_test, prediction, average="weighted")), 4),
        }
    )

classification_table = pd.DataFrame(classification_results).sort_values("accuracy", ascending=False)
classification_table

,model,accuracy,f1_weighted
2,Neural Network,0.9466,0.9466
1,KNN,0.9335,0.9333
0,Naive Bayes,0.8298,0.8141


In [5]:
best_reg_name = regression_table.iloc[0]["model"]
best_clf_name = classification_table.iloc[0]["model"]
best_regressor = regressors[best_reg_name]
best_classifier = classifiers[best_clf_name]

best_regressor.fit(X, y_reg)
best_classifier.fit(X, y_clf)

metadata = {
    "features": FEATURES,
    "choices": {
        "city": sorted(df["city"].unique().tolist()),
        "furnishing": sorted(df["furnishing"].unique().tolist()),
    },
    "categories": ["affordable", "average", "expensive"],
    "currency": "KES",
    "thresholds": {"affordable_max": round(float(q33), 2), "average_max": round(float(q66), 2)},
    "selected_models": {"regressor": best_reg_name, "classifier": best_clf_name},
    "comparison": {"regression": regression_results, "classification": classification_results},
    "training_rows": int(len(df)),
}

joblib.dump(best_regressor, MODEL_DIR / "rent_regressor.joblib")
joblib.dump(best_classifier, MODEL_DIR / "rent_classifier.joblib")
(MODEL_DIR / "columns.json").write_text(json.dumps(metadata, indent=2))

print("Saved:", best_reg_name, "and", best_clf_name)
print(json.dumps(metadata["comparison"], indent=2))

Saved: Neural Network and Neural Network
{
  "regression": [
    {
      "model": "Ridge",
      "r2": 0.9234,
      "mae": 14315.05,
      "rmse": 29315.62
    },
    {
      "model": "Lasso",
      "r2": 0.9312,
      "mae": 13848.99,
      "rmse": 27777.64
    },
    {
      "model": "KNN",
      "r2": 0.9287,
      "mae": 15154.67,
      "rmse": 28288.8
    },
    {
      "model": "Neural Network",
      "r2": 0.9495,
      "mae": 13446.33,
      "rmse": 23788.9
    }
  ],
  "classification": [
    {
      "model": "Naive Bayes",
      "accuracy": 0.8298,
      "f1_weighted": 0.8141
    },
    {
      "model": "KNN",
      "accuracy": 0.9335,
      "f1_weighted": 0.9333
    },
    {
      "model": "Neural Network",
      "accuracy": 0.9466,
      "f1_weighted": 0.9466
    }
  ]
}
